In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils import shuffle
from sklearn import svm
import pyswarms as ps
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from deap import algorithms, base, creator, tools
# Read data
data = pd.read_excel('聚类分析2.xlsx')  # Please replace with your data file name
data = data.drop('合金的牌号', axis=1)
data = data.round(2)
elements = ['Ni', 'Cr', 'Co', 'Fe', 'Al', 'Ti', 'Nb', 'Mo', 'W', 'C', 'B', 'Zr', '温度', '应力', '固溶处理温度',
            '固溶处理时间', '强化相溶解温度', '稳定时效温度', '稳定时效时间', '时效温度', '时效时间']

X = np.zeros((data.shape[0], len(elements)))
for i, row in data.iterrows():
    for j, el in enumerate(elements):
        X[i, j] = row[el]
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
Y = data[['蠕变时间']].values
Y = np.log(Y)
def evaluate(true_labels, pred_labels):
    errors = abs(pred_labels - true_labels)
    MAE = mean_absolute_error(true_labels, pred_labels)
    mape = 100 * np.mean(errors / true_labels)
    r2 = r2_score(true_labels, pred_labels)
    RMSE = mean_squared_error(true_labels, pred_labels, squared=False)
    return mape, MAE, RMSE, r2
model_0 = SVR(C=37.722551111111111111, kernel='rbf', gamma=0.111111111111111111)
cv = LeaveOneOut()
Model = model_0
X = X.reshape(-1, len(elements))
Model.fit(X, Y)
from sklearn.model_selection import cross_val_predict
y_pred = cross_val_predict(Model, X, Y, cv=cv)
model_accuracy1, model_MAE1, model_RMSE1, model_r21 = evaluate(Y, y_pred)
print("R^2 score:", model_r21)
def fitness_func(individual, X, Y, model):
    X_individual = np.tile(individual, (X.shape[0], 1))
    Y_pred = model.predict(X_individual)
    mse = mean_squared_error(Y, Y_pred)
    return (1 / (1 + mse),)
# Define GA parameters
POP_SIZE = 400
NUM_GEN = 100
CXPB = 0.9
MUTPB = 0.01
# Create a fitness function for maximizing fitness values
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
# Initialize toolbox
toolbox = base.Toolbox()
# Define attributes
toolbox.register("attr_float", np.random.uniform, low=0, high=1)
# Define individual and population
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=len(elements))
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
# Define evaluation function
toolbox.register("evaluate", fitness_func, X=X, Y=Y, model=Model)
# Define genetic operators
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
# Perform genetic algorithm
population = toolbox.population(n=POP_SIZE)
hall_of_fame = tools.HallOfFame(maxsize=1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
population, logbook = algorithms.eaSimple(population, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=NUM_GEN,
                                          stats=stats, halloffame=hall_of_fame, verbose=True)
# Extract best individual and its fitness
best_individual = hall_of_fame[0]
best_fitness = best_individual.fitness.values[0]
# Reshape best_individual
best_particles = np.asarray(best_individual)
print("Best particles:", best_particles)
print("Best fitness:", best_fitness)

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

R^2 score: 0.8804390625989749
gen	nevals	avg     	min     
0  	400   	0.335418	0.107702
1  	378   	0.370719	0.114245
2  	368   	0.387392	0.191028
3  	364   	0.388014	0.154489
4  	360   	0.38716 	0.222551
5  	364   	0.388669	0.200006
6  	356   	0.38986 	0.172766
7  	364   	0.392695	0.195595
8  	360   	0.39231 	0.234139
9  	376   	0.394686	0.25074 
10 	362   	0.390942	0.164721
11 	359   	0.391965	0.218376
12 	360   	0.392569	0.237674
13 	372   	0.395845	0.219809
14 	359   	0.396828	0.244195
15 	367   	0.396053	0.223302
16 	352   	0.398451	0.247524
17 	362   	0.400721	0.244364
18 	338   	0.400467	0.260462
19 	362   	0.399123	0.23669 
20 	367   	0.399782	0.264399
21 	374   	0.403673	0.232776
22 	370   	0.40026 	0.253117
23 	364   	0.404279	0.249674
24 	372   	0.402356	0.22489 
25 	368   	0.404952	0.150873
26 	364   	0.405599	0.28073 
27 	349   	0.408222	0.279731
28 	370   	0.407387	0.282893
29 	358   	0.408911	0.33543 
30 	361   	0.41025 	0.356896
31 	372   	0.408388	0.287565
32 	376   	0.

# 输出表格中X对应数据

In [2]:
X=X.round(2)
X_normalized=X
for i in range(X_normalized.shape[1]):
    column = X_normalized[:, i]
    print("Column", i+1, ":", column)
    print("---------------------------------------------------------------------------------------")

Column 1 : [0.51 0.19 0.19 0.17 0.25 0.41 0.24 0.43 0.36 0.29 0.51 0.36 0.34 0.44
 0.37 0.12 0.6  0.58 0.71 0.27 0.83 0.91 0.98 1.   0.49 0.19 0.47 0.41
 0.37 0.   0.31 0.85 0.6  0.54 0.54 0.66 0.45 0.53 0.73 0.77]
---------------------------------------------------------------------------------------
Column 2 : [0.88 0.91 0.91 0.82 0.83 0.82 0.82 0.79 0.8  0.84 0.81 0.87 0.81 0.73
 0.77 0.75 0.75 0.75 0.75 1.   0.84 0.36 0.17 0.   0.35 0.81 0.79 0.81
 0.59 0.81 0.81 0.81 0.71 0.71 0.71 0.71 0.69 0.71 0.9  0.91]
---------------------------------------------------------------------------------------
Column 3 : [0.43 0.43 0.43 1.   1.   1.   1.   1.   1.   1.   1.   1.   1.   1.
 0.4  0.4  0.4  0.4  0.4  0.5  0.5  0.   0.   0.   0.   0.51 0.51 0.51
 0.51 0.51 0.51 0.51 0.3  0.3  0.3  0.3  0.3  0.3  0.43 0.58]
---------------------------------------------------------------------------------------
Column 4 : [0.99 1.   1.   0.08 0.16 0.12 0.12 0.22 0.13 0.11 0.13 0.14 0.15 0.12
 0.43 0.45 

In [3]:
import pandas as pd
import numpy as np
X = X.round(2)
X_normalized=X
df = pd.DataFrame()
for i in range(X_normalized.shape[1]):
    column = X_normalized[:, i]
    column_name = "Column " + str(i+1)
    df[column_name] = column
df.to_excel('output.xlsx', index=False)

# 设计新合金

# 利用for循环进行数据的比对和输出

In [16]:
import pandas as pd
df_particles = pd.read_excel('best_particles.xlsx')
df_particles = df_particles.round(2)
df_output = pd.read_excel('output.xlsx')
df_output = df_output.round(2)
num_columns = len(df_particles.columns)
for i in range(num_columns):
    column_particles = df_particles.iloc[:, i]
    column_output = df_output.iloc[:, i]
    column_name = column_particles.name
    for j in range(len(column_particles)):
        value_particles = column_particles.iloc[j]
        for k in range(len(column_particles)):
            value_output = column_output.iloc[k]
            diff = abs(value_particles - value_output)
            if diff <= 0.01:
                print("-------------------------------------------")
                print("Column:", column_name)
                print("Position (Row, Column):", k+1, i+1)
                print("Value (best_particles):", value_particles)
                print("Value (output):", value_output)
                print("-------------------------------------------")

Column: 0.6357775635235999
Position (Row, Column): 7 1
Value (best_particles): 0.43
Value (output): 0.43
-------------------------------------------
Column: 0.6357775635235999
Position (Row, Column): 8 1
Value (best_particles): 0.36
Value (output): 0.36
-------------------------------------------
Column: 0.6357775635235999
Position (Row, Column): 11 1
Value (best_particles): 0.36
Value (output): 0.36
-------------------------------------------
Column: 0.6357775635235999
Position (Row, Column): 31 1
Value (best_particles): 0.85
Value (output): 0.85
-------------------------------------------
Column: 0.6357775635235999
Position (Row, Column): 5 1
Value (best_particles): 0.41
Value (output): 0.41
-------------------------------------------
Column: 0.6357775635235999
Position (Row, Column): 27 1
Value (best_particles): 0.41
Value (output): 0.41
-------------------------------------------
Column: 0.6357775635235999
Position (Row, Column): 4 1
Value (best_particles): 0.25
Value (output): 0.2

Column: 0.1801821248293553
Position (Row, Column): 25 18
Value (best_particles): 0.01
Value (output): 0.0
-------------------------------------------
Column: 0.1801821248293553
Position (Row, Column): 30 18
Value (best_particles): 0.95
Value (output): 0.95
-------------------------------------------
Column: 0.1801821248293553
Position (Row, Column): 1 18
Value (best_particles): 0.98
Value (output): 0.98
-------------------------------------------
Column: 0.1801821248293553
Position (Row, Column): 2 18
Value (best_particles): 0.98
Value (output): 0.98
-------------------------------------------
Column: 0.1801821248293553
Position (Row, Column): 3 18
Value (best_particles): 0.98
Value (output): 0.98
-------------------------------------------
Column: 0.1801821248293553
Position (Row, Column): 4 18
Value (best_particles): 0.98
Value (output): 0.98
-------------------------------------------
Column: 0.1801821248293553
Position (Row, Column): 5 18
Value (best_particles): 0.98
Value (output)

# 将上述的输出结果保存在表格中

In [18]:
import pandas as pd
df_particles = pd.read_excel('best_particles.xlsx')
df_particles = df_particles.round(2)
df_output = pd.read_excel('output.xlsx')
df_output = df_output.round(2)
num_columns = len(df_particles.columns)
results = pd.DataFrame(columns=['Column', 'Position (Row, Column)', 'Value (best_particles)', 'Value (output)'])
for i in range(num_columns):
    column_particles = df_particles.iloc[:, i]
    column_output = df_output.iloc[:, i]
    column_name = column_particles.name 
    for j in range(len(column_particles)):
        value_particles = column_particles.iloc[j]
        for k in range(len(column_output)):
            value_output = column_output.iloc[k]
            diff = abs(value_particles - value_output)
            if diff <= 0.01:
                row = {
                    'Column': column_name,
                    'Position (Row, Column)': f'{k+1}, {i+1}',
                    'Value (best_particles)': value_particles,
                    'Value (output)': value_output
                }
                results = pd.concat([results, pd.DataFrame(row, index=[0])], ignore_index=True)
results.to_excel('results.xlsx', index=False)

# 将表格中的相对应的行和列数据还原出真实的数据

In [20]:
import pandas as pd
df_julie = pd.read_excel('julie.xlsx')
df_results = pd.read_excel('results.xlsx')
df_results['Julie Data'] = ''
for index, row in df_results.iterrows():
    row_num, col_num = map(int, row['Position (Row, Column)'].split(','))  # 获取行号和列号
    julie_data = df_julie.iloc[row_num-1, col_num-1]  # 根据行号和列号获取Julie.xls数据库中对应的数据
    df_results.at[index, 'Julie Data'] = julie_data  # 将Julie数据添加到结果表格中
df_results.to_excel('final_results.xlsx', index=False)

# 将数据进行一定的筛选

In [23]:
import pandas as pd
df = pd.read_excel('final_results.xlsx')
df.drop_duplicates(subset='Julie Data', keep='first', inplace=True)
df.to_excel('final_results_no_duplicates.xlsx', index=False)

In [24]:
import pandas as pd
df = pd.read_excel('final_results.xlsx')
df = df.round(2)
df.drop_duplicates(subset='Julie Data', keep='first', inplace=True)
df.to_excel('final_results_no_duplicates1.xlsx', index=False)